# Main Pipeline Scan Plan

Forward-simulating planner for the `radio` script's brick-interleave galactic-plane
survey.  Uses measured cadence + slew statistics from `labs/04/data/archive/main`
to project which cells will be reachable when the scan reaches them.

This mirrors `radio`'s `main()` logic so notebook and script make the same plan.

In [ ]:
import math
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.coordinates as ac
from astropy.time import Time, TimeDelta

import ugradiolab.plotting as plotting
from ugradiolab.astronomy import (
    LEO_LAT_DEG, LEO_LON_DEG, LEO_OBS_ALT_M,
    compute_gal_pointing,
)
from utils.timing_stats import load as load_timing_stats

%matplotlib inline

# Match radio's config
L_CENTER = 120.0
L_MIN, L_MAX = -10.0, 250.0
B_MIN, B_MAX = -6, 6
B_STEP = 2
PHYSICAL_SPACING_DEG = 2.0

MIN_ALT_DEG = 17.0
MAX_ALT_DEG = 83.0
AZ_MIN, AZ_MAX = 7.0, 348.0

stats = load_timing_stats('artifacts/main_timing_stats.json',
                         archive_dir='data/archive/main')
CELL_TIME_P50 = stats['cell_total_time_sec']['p50']
print(f"timing stats: {stats['n_cells_observed']} cells across "
      f"{stats['n_sessions']} sessions")
print(f"  intra-cell cadence: p50 {stats['intra_cell_cadence_sec']['p50']:.1f}s, "
      f"p95 {stats['intra_cell_cadence_sec']['p95']:.1f}s")
print(f"  slew gap:           p50 {stats['slew_gap_sec']['p50']:.1f}s, "
      f"p95 {stats['slew_gap_sec']['p95']:.1f}s")
print(f"  cell total:         p50 {CELL_TIME_P50:.1f}s, "
      f"p95 {stats['cell_total_time_sec']['p95']:.1f}s")
print(f"  duty cycle:         {stats['duty_cycle']:.2f}")

## 1. Build the brick-interleave grid

Mirrors `build_galplane_grid` from `radio`.  Even and odd phases are concatenated.

In [ ]:
def build_l_row(b_deg, l_center=L_CENTER):
    cos_b = math.cos(math.radians(b_deg))
    if cos_b <= 0:
        return [l_center]
    dl = PHYSICAL_SPACING_DEG / cos_b
    l_vals = [round(l_center, 2)]
    l = l_center + dl
    while l <= L_MAX:
        l_vals.append(round(l, 2))
        l += dl
    l = l_center - dl
    while l >= L_MIN:
        l_vals.append(round(l, 2))
        l -= dl
    return sorted(l_vals)


def build_galplane_grid(phase='even'):
    if phase == 'even':
        b_vals = list(range(B_MIN, B_MAX + 1, B_STEP))
    else:
        b_vals = list(range(B_MIN + 1, B_MAX, B_STEP))
    all_cells = []
    for b in b_vals:
        if phase == 'odd':
            half_step = PHYSICAL_SPACING_DEG / (2 * math.cos(math.radians(b)))
            l_center = L_CENTER + half_step
        else:
            l_center = L_CENTER
        for l in build_l_row(b, l_center=l_center):
            all_cells.append((l, b))
    all_cells.sort(key=lambda c: c[0])
    col_tol = PHYSICAL_SPACING_DEG / 2
    columns = [[all_cells[0]]]
    for cell in all_cells[1:]:
        if cell[0] - columns[-1][0][0] <= col_tol:
            columns[-1].append(cell)
        else:
            columns.append([cell])
    cells = []
    for col_idx, col in enumerate(columns):
        col_sorted = sorted(col, key=lambda c: c[1])
        if col_idx % 2 == 1:
            col_sorted = list(reversed(col_sorted))
        for row_idx, (l, b) in enumerate(col_sorted):
            cells.append((col_idx, row_idx, l, b))
    return cells


even_grid = build_galplane_grid(phase='even')
odd_grid = build_galplane_grid(phase='odd')
full_grid = even_grid + odd_grid
print(f'even phase: {len(even_grid)} cells')
print(f'odd phase:  {len(odd_grid)} cells')
print(f'total:      {len(full_grid)} cells')

## 2. Pick az side and forward-simulate

For each cell, project its observation time as `now + i * cell_total_time_sec`,
evaluate alt/az at that projected time, and keep cells inside the limits.

In [ ]:
def filter_by_az_side(cells):
    classified = []
    for row, col, l, b in cells:
        alt, az, ra, dec, _ = compute_gal_pointing(
            l, b, lat=LEO_LAT_DEG, lon=LEO_LON_DEG, obs_alt=LEO_OBS_ALT_M,
        )
        max_alt = 90.0 - abs(LEO_LAT_DEG - dec)
        if max_alt < MIN_ALT_DEG:
            continue
        in_limits = MIN_ALT_DEG <= alt <= MAX_ALT_DEG
        is_rising = (AZ_MIN <= az <= 180) or az > AZ_MAX or az < AZ_MIN
        is_setting = 180 < az <= AZ_MAX
        classified.append((row, col, l, b, alt, az, in_limits, is_rising, is_setting))
    n_rising = sum(1 for c in classified if c[6] and c[7])
    n_setting = sum(1 for c in classified if c[6] and c[8])
    side = 'rising' if n_rising >= n_setting else 'setting'
    if side == 'rising':
        kept = [(r, c, l, b) for r, c, l, b, *_, rising, setting in classified if rising]
    else:
        kept = [(r, c, l, b) for r, c, l, b, *_, rising, setting in classified if setting]
    return kept, side, n_rising, n_setting


def forward_simulate(cells, cell_time_sec, index_offset=0):
    now = time.time()
    rows = []
    for i, (_, _, l, b) in enumerate(cells):
        proj_t = now + (index_offset + i + 0.5) * cell_time_sec
        alt, az, ra, dec, _ = compute_gal_pointing(
            l, b, lat=LEO_LAT_DEG, lon=LEO_LON_DEG, obs_alt=LEO_OBS_ALT_M,
            unix_t=proj_t,
        )
        in_alt = MIN_ALT_DEG <= alt <= MAX_ALT_DEG
        in_az = AZ_MIN <= az <= AZ_MAX
        rows.append({
            'l': l, 'b': b, 'proj_t': proj_t,
            'alt': alt, 'az': az,
            'in_alt': in_alt, 'in_az': in_az,
            'kept': in_alt and in_az,
        })
    return rows


even_side, side_e, ne_r, ne_s = filter_by_az_side(even_grid)
odd_side, side_o, no_r, no_s = filter_by_az_side(odd_grid)
print(f'even: side={side_e} (rising {ne_r} / setting {ne_s}); kept {len(even_side)}')
print(f'odd:  side={side_o} (rising {no_r} / setting {no_s}); kept {len(odd_side)}')

even_sim = forward_simulate(even_side, CELL_TIME_P50, index_offset=0)
odd_sim = forward_simulate(odd_side, CELL_TIME_P50, index_offset=len(even_side))
all_sim = even_sim + odd_sim
n_kept = sum(1 for r in all_sim if r['kept'])
n_drop_alt = sum(1 for r in all_sim if not r['in_alt'])
n_drop_az = sum(1 for r in all_sim if r['in_alt'] and not r['in_az'])
print(f'forward sim ({CELL_TIME_P50:.0f}s/cell): kept {n_kept}/{len(all_sim)}, '
      f'dropped {n_drop_alt} alt, {n_drop_az} az')
if n_kept > 0:
    last_kept = max(i for i, r in enumerate(all_sim) if r['kept'])
    horizon_h = (last_kept + 1) * CELL_TIME_P50 / 3600
    print(f'planning horizon: {horizon_h:.1f}h to last reachable cell')

## 3. Visualise coverage

Galactic-coordinate scatter coloured by projected hours-from-now.  Dropped cells
are shown in grey.

In [ ]:
now = time.time()
kept = [r for r in all_sim if r['kept']]
drop = [r for r in all_sim if not r['kept']]

fig, ax = plt.subplots(figsize=(plotting.TEXTWIDTH_IN, 4))
if drop:
    ax.scatter([r['l'] for r in drop], [r['b'] for r in drop],
               c='lightgrey', s=plotting.SS_FINE, edgecolors='none',
               alpha=0.5, label=f'dropped ({len(drop)})')
if kept:
    hours = [(r['proj_t'] - now) / 3600 for r in kept]
    sc = ax.scatter([r['l'] for r in kept], [r['b'] for r in kept],
                    c=hours, cmap='viridis', s=plotting.SS_FINE,
                    edgecolors='none', label=f'kept ({len(kept)})')
    cbar = plt.colorbar(sc, ax=ax, shrink=0.8)
    cbar.set_label('Projected hours from now')
ax.set_xlabel(r'$\ell$ [deg]')
ax.set_ylabel(r'$b$ [deg]')
ax.set_title(f'Forward-simulated plan ({CELL_TIME_P50:.0f}s/cell)')
ax.set_aspect('equal')
ax.grid(True, **plotting.GRID_STYLE)
ax.legend(fontsize=plotting.LEGEND_SIZE, loc='upper right')
fig.tight_layout()
plt.show()

## 4. Alt/az timeline

alt and az traces of all kept cells in scan order, with the alt/az exclusion
limits marked.  Useful for spotting tight margins or runs that exit the limits
near the end.

In [ ]:
if kept:
    hours = np.array([(r['proj_t'] - now) / 3600 for r in kept])
    alt_arr = np.array([r['alt'] for r in kept])
    az_arr = np.array([r['az'] for r in kept])

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(plotting.TEXTWIDTH_IN, 5), sharex=True)
    ax1.plot(hours, alt_arr, '.', ms=plotting.MS_FINE, color='C0')
    ax1.axhline(MIN_ALT_DEG, ls='--', color='C3', alpha=plotting.ALPHA_LIGHT)
    ax1.axhline(MAX_ALT_DEG, ls='--', color='C3', alpha=plotting.ALPHA_LIGHT)
    ax1.set_ylabel('alt [deg]')
    ax1.grid(True, **plotting.GRID_STYLE)

    ax2.plot(hours, az_arr, '.', ms=plotting.MS_FINE, color='C1')
    ax2.axhline(AZ_MIN, ls='--', color='C3', alpha=plotting.ALPHA_LIGHT)
    ax2.axhline(AZ_MAX, ls='--', color='C3', alpha=plotting.ALPHA_LIGHT)
    ax2.set_xlabel('hours from now')
    ax2.set_ylabel('az [deg]')
    ax2.grid(True, **plotting.GRID_STYLE)
    fig.suptitle('alt/az trace of kept cells')
    fig.tight_layout()
    plt.show()
else:
    print('no kept cells -- nothing to plot')

## 5. Summary

Cell counts before/after each filter, plus projected total run time.

In [ ]:
n_total = len(full_grid)
n_after_az = len(even_side) + len(odd_side)
n_after_fwd = n_kept
wall_time_h = n_after_fwd * CELL_TIME_P50 / 3600
cap_time_h = wall_time_h * stats['duty_cycle']
print(f'  full grid:           {n_total}')
print(f'  after az-side:       {n_after_az}  '
      f'(even {side_e} / odd {side_o})')
print(f'  after forward-sim:   {n_after_fwd}')
print(f'  projected wallclock: {wall_time_h:.1f}h')
print(f'  projected on-sky:    {cap_time_h:.1f}h')